# 03 — Evaluate and infer

Fit per-(type, K-bucket) temperature on a held-out slice of the train split, then
report raw and post-temperature ECE / Brier / NLL / accuracy on the test split,
and score individual questions.


In [ ]:
from pathlib import Path
import os, sys

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)


## 1. Pick a checkpoint


In [ ]:
from pathlib import Path

available = sorted(Path("checkpoints").glob("*/*.pt"))
print("available checkpoints:")
print("\n".join(str(p) for p in available) or "(none yet — run 01 or 02 first)")

CKPT = Path("checkpoints/rlcd_smoke/best.pt")  # <- edit to the run you want
assert CKPT.exists(), f"checkpoint not found: {CKPT}"
CKPT


## 2. Full evaluation

Mirrors `scripts/training/evaluate.py`: the model is rebuilt from the architecture
stored in the checkpoint, so no YAML is needed.


In [ ]:
from src.pipelines.config import TrainingConfig
from src.pipelines.eval import evaluate, format_report
from src.pipelines.infer import load_model
from src.pipelines.train import build_loaders
from src.utils.model_utils import detect_device

device = detect_device()
model, model_cfg, tokenizer = load_model(CKPT, device)

train_cfg = TrainingConfig(calib_fraction=0.1, batch_size=16, seed=42)
_, calib_loader, test_loader, _ = build_loaders(train_cfg, model_cfg, tokenizer, device)
assert calib_loader is not None, "calib_fraction must be > 0 to fit temperature"

result = evaluate(model, calib_loader, test_loader, device)
print(format_report(result))


In [ ]:
import json
print(json.dumps(result, indent=2))


## 3. Score a single question


In [ ]:
from src.pipelines.infer import infer

state = "Agent ran 3 tool calls, one returned HTTP 500, then retried successfully."

choice = {
    "type": "choice",
    "instructions": "What should the system do next?",
    "criteria": {"continue": "proceed with the plan", "escalate": "hand to a human"},
}
infer(CKPT, state, choice)


In [ ]:
score = {
    "type": "score",
    "instructions": "How severe is the incident?",
    "criteria": ["cosmetic", "minor", "major", "critical"],
}
infer(CKPT, state, score)


In [ ]:
noul = {
    "type": "noul",
    "instructions": "Did any tool call fail?",
}
infer(CKPT, state, noul)
